# Exercices · Tests d'hypothèses et tests A/B

Exercices réalisés pendant le bootcamp Data Essentials (Jedha, 2026). Les jeux de données ont été fournis par le bootcamp et ne sont pas inclus dans le dépôt.

**Démarche commune à chaque test :**
1. Formuler l'hypothèse nulle H0 (le statu quo, ce qu'on cherche à contredire) et l'hypothèse alternative H1 (ce qu'on veut démontrer).
2. Fixer le risque d'erreur accepté : α = 5 %.
3. Calculer la statistique de test et la **p-value**.
4. Décider : si p-value < α, on rejette H0 ; sinon, on ne peut pas la rejeter.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.weightstats import ztest

ALPHA = 0.05

# Dossier contenant les fichiers de données fournis par le bootcamp
# Sur Google Colab : téléverser les fichiers puis utiliser DATA_DIR = "/content/"
DATA_DIR = "data/"

### Visualiser la p-value

La courbe représente la distribution de la statistique de test **si H0 était vraie**. La zone rouge est la p-value : la probabilité d'observer, par simple hasard, un résultat au moins aussi extrême que celui obtenu. Plus elle est petite, moins il est plausible que le résultat soit dû au hasard.

In [ ]:
def afficher_p_value(z_obs, alternative):
    """Trace la loi normale centrée réduite et colorie la zone correspondant à la p-value."""
    x = np.linspace(-4, 4, 1000)
    y = stats.norm.pdf(x)
    zone = x <= z_obs if alternative == "smaller" else x >= z_obs

    plt.figure(figsize=(7, 3.5))
    plt.plot(x, y, color="black")
    plt.fill_between(x[zone], y[zone], color="red", alpha=0.6, label="p-value")
    plt.axvline(z_obs, color="red", linestyle="--", label=f"z observé = {z_obs:.2f}")
    plt.title("Distribution de la statistique sous H0")
    plt.legend()
    plt.show()

## Exercice 1 · Qonto

**Question :** le taux de clients en défaut de paiement (`TARGET = 1`) est-il inférieur à 8,5 % ?

- H0 : taux ≥ 8,5 %
- H1 : taux < 8,5 % (test unilatéral à gauche)

In [ ]:
qonto = pd.read_csv(DATA_DIR + "qonto_loans.csv")

taux_observe = qonto["TARGET"].mean()
print(f"{len(qonto)} clients, taux observé : {taux_observe:.2%}")

Calcul manuel de la statistique z, puis vérification avec `statsmodels` :

In [ ]:
mu_0 = 0.085
n = len(qonto)

z_manuel = (taux_observe - mu_0) / (qonto["TARGET"].std() / np.sqrt(n))
z_score, p_value = ztest(qonto["TARGET"], value=mu_0, alternative="smaller")

print(f"z (calcul manuel) : {z_manuel:.3f}")
print(f"z (statsmodels)   : {z_score:.3f}")
print(f"p-value           : {p_value:.4f}")
print("On rejette H0" if p_value < ALPHA else "On ne peut pas rejeter H0")

In [ ]:
afficher_p_value(z_score, alternative="smaller")

**Conclusion :** avec une p-value d'environ 0,003, bien inférieure à 5 %, on rejette H0. Le taux de défaut observé (7,75 %) est **significativement inférieur à 8,5 %**.

## Exercice 2 · Swile

**Question :** les commerciaux passent-ils plus d'appels en France qu'à l'étranger ?

- H0 : différence moyenne (France − étranger) ≤ 0
- H1 : différence moyenne > 0 (test unilatéral à droite)

In [ ]:
swile = pd.read_csv(DATA_DIR + "swile_sales_calls.csv")
swile.describe().round(2)

In [ ]:
swile.mean().plot(kind="bar", title="Nombre moyen d'appels en France et à l'étranger", rot=0)
plt.ylabel("Appels")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, colonne in zip(axes, ["calls_in_France", "calls_abroad"]):
    ax.hist(swile[colonne], bins=10)
    ax.set_title(colonne)
    ax.set_xlabel("Nombre d'appels")
axes[0].set_ylabel("Fréquence")
plt.tight_layout()
plt.show()

**Quel test choisir ?** Si chaque ligne correspond au **même commercial** (ses appels en France et à l'étranger), les deux mesures sont appariées : on teste la moyenne des différences avec un test t sur un échantillon. Si les deux colonnes correspondent à **deux groupes de commerciaux différents**, il faut un test pour échantillons indépendants (test de Welch). On calcule les deux pour vérifier que la conclusion ne dépend pas de ce choix.

In [ ]:
diff = swile["calls_in_France"] - swile["calls_abroad"]

t_apparie, p_apparie = stats.ttest_1samp(diff, popmean=0, alternative="greater")
t_welch, p_welch = stats.ttest_ind(
    swile["calls_in_France"], swile["calls_abroad"], equal_var=False, alternative="greater"
)

print(f"Test apparié : t = {t_apparie:.3f}, p-value = {p_apparie:.3f}")
print(f"Test de Welch : t = {t_welch:.3f}, p-value = {p_welch:.3f}")
print("On rejette H0" if p_apparie < ALPHA else "On ne peut pas rejeter H0")

**Conclusion :** dans les deux cas, la p-value (environ 0,24) est **supérieure au seuil α = 5 %** : on ne peut pas rejeter H0. Les données ne permettent pas d'affirmer que les commerciaux passent plus d'appels en France qu'à l'étranger. La petite différence observée (5,8 contre 5,5 appels en moyenne) peut s'expliquer par le hasard.

## Exercice 3 · Vinted

**Question :** la fonctionnalité B donne-t-elle un meilleur taux de succès que la fonctionnalité A ? Chaque version a été testée sur 500 utilisateurs (1 = succès, 0 = échec).

- H0 : taux B ≤ taux A
- H1 : taux B > taux A (test unilatéral à droite)

In [ ]:
vinted = pd.read_csv(DATA_DIR + "vinted_website_optimization.csv")

taux = vinted.mean()
n_v = len(vinted)
marges = stats.norm.ppf(0.975) * vinted.std() / np.sqrt(n_v)   # IC à 95 % de chaque taux

print(taux.round(3))

taux.plot(kind="bar", yerr=marges, capsize=6, rot=0, title="Taux de succès (IC à 95 %)")
plt.ylabel("Taux de succès")
plt.show()

Calcul manuel de la statistique z, puis vérification avec `statsmodels` :

In [ ]:
mean_a, mean_b = vinted["Feature_A"].mean(), vinted["Feature_B"].mean()
std_a, std_b = vinted["Feature_A"].std(), vinted["Feature_B"].std()

z_manuel = (mean_b - mean_a) / np.sqrt(std_a**2 / n_v + std_b**2 / n_v)
z_score, p_value = ztest(vinted["Feature_B"], vinted["Feature_A"], value=0, alternative="larger")

print(f"z (calcul manuel) : {z_manuel:.3f}")
print(f"z (statsmodels)   : {z_score:.3f}")
print(f"p-value           : {p_value:.4f}")
print("On rejette H0" if p_value < ALPHA else "On ne peut pas rejeter H0")

In [ ]:
afficher_p_value(z_score, alternative="larger")

**Conclusion :** avec une p-value d'environ 0,02, inférieure à 5 %, on rejette H0. La fonctionnalité B obtient un taux de succès **significativement supérieur** à A (12,6 % contre 8,6 %, soit +4 points). D'un point de vue business, B peut être déployée, en gardant en tête que les intervalles de confiance des deux taux se recouvrent en partie : un test sur plus d'utilisateurs préciserait l'ampleur du gain.